 # B4 — Taxonomy development



 Two passes:



 **Pass 1 (cells 1–3):** draw a 150-review reading sample and export a tally

 sheet. Then leave the notebook, read all 150, and fill the sheet in.



 **Pass 2 (cells 4–6):** load the filled sheet and look at label frequency,

 co-occurrence, and what did not fit. Refine `taxonomy.py` from that.



 These 150 reviews are **excluded from the gold set in B5**. Categories drawn

 to fit specific reviews would make agreement on those same reviews look better

 than it is.



 Only aggregates are printed here. Review text goes to a local file.

In [8]:
from pathlib import Path

import pandas as pd

from support_reply_gap.analysis.data import load_reviews
from support_reply_gap.config import ANALYSIS_WINDOW_START
from support_reply_gap.labeling.sampling import (
    DEV_SAMPLE_PER_SCORE,
    add_language,
    add_text_flags,
    drop_repeated_text,
    eligible_for_labelling,
    stratified_sample,
)
from support_reply_gap.labeling.taxonomy import LABEL_NAMES

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 220)

# Jupyter runs from notebooks/; anchor paths at the repo root so .gitignore covers data/dev/.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# Not committed: contains review text.
TALLY_PATH = ROOT / "data/dev/taxonomy_tally.csv"
# Committed: opaque ids only, so B5 can reproducibly exclude them.
DEV_IDS_PATH = ROOT / "data/gold/dev_sample_ids.txt"

 ## 1. Draw the sample

In [9]:
df = load_reviews(since=ANALYSIS_WINDOW_START)
print(f"{len(df):,} reviews in the analysis window")

df = add_text_flags(df)
df = add_language(df)
eligible = drop_repeated_text(eligible_for_labelling(df))
print(f"{len(eligible):,} eligible after the B3 cleaning rules")
print(eligible.groupby(["app_key", "score"]).size().unstack(fill_value=0))


4,990 reviews in the analysis window
3,789 eligible after the B3 cleaning rules
score      1   2   3    4     5
app_key                        
c24      161  87  69   74   185
n26      266  52  54   54   319
revolut  412  48  57  181  1770


In [10]:
sample = stratified_sample(eligible, per_score=DEV_SAMPLE_PER_SCORE)
print(f"{len(sample)} drawn")
print(pd.crosstab(sample["app_key"], sample["score"]))


150 drawn
score     1   2   3  4  5
app_key                  
c24      16  14  13  5  5
n26      17  13  12  5  5
revolut  17   8  10  5  5


## 2. Export the tally sheet

One row per review, one empty column per label. Enter `1` to apply a label
and `0` otherwise. Every label cell must be filled; the loader rejects
anything else.

Two further columns matter more than the label columns:

- `doesnt_fit` — `1` if the review carries a complaint none of the eight
  labels covers, `0` otherwise. Describe the gap in `notes`.
- `notes` — what did not fit, and anything that made the call hard:
  ambiguity, two readings, a boundary between two labels that is not where
  the taxonomy puts it.

`utf-8-sig` so Excel on macOS opens umlauts correctly.

In [11]:
# Never overwrite: the sheet holds hand labels, and the ids must match it.
if TALLY_PATH.exists() or DEV_IDS_PATH.exists():
    print(f"skipped export: {TALLY_PATH.name} or {DEV_IDS_PATH.name} already exists")
else:
    TALLY_PATH.parent.mkdir(parents=True, exist_ok=True)
    DEV_IDS_PATH.parent.mkdir(parents=True, exist_ok=True)

    tally = sample[["review_id", "app_key", "score", "posted_at", "content"]].copy()
    tally["posted_at"] = tally["posted_at"].dt.date
    for name in LABEL_NAMES:
        tally[name] = ""
    tally["doesnt_fit"] = ""
    tally["notes"] = ""

    tally.to_csv(TALLY_PATH, index=False, encoding="utf-8-sig")
    DEV_IDS_PATH.write_text("\n".join(sorted(sample["review_id"])) + "\n", encoding="utf-8")

    print(f"wrote {TALLY_PATH}  ({len(tally)} rows)")
    print(f"wrote {DEV_IDS_PATH}")

skipped export: taxonomy_tally.csv or dev_sample_ids.txt already exists


 ## 3. Stop here and read



 Open `data/dev/taxonomy_tally.csv` in Numbers or Excel and work through all

 150. Roughly 60–90 minutes.



 Four hypotheses worth testing specifically — each is a plausible gap in the

 current eight labels, and each is cheap to confirm or reject while reading:



 1. **Account closure / frozen funds.** "Account closed without warning",

    "money frozen for weeks". Fits neither `login_access` (locked out of a

    working account) nor `kyc_onboarding` (could not open one). Common in

    neobank reviews and the most likely genuine gap.

 2. **Trading and crypto.** Revolut is a banking *and trading* app; N26 and C24

    are not. Order execution, crypto withdrawals, portfolio display. If these

    appear only for Revolut, a label for them would be app-specific — which is

    an argument for folding them in rather than adding a class.

 3. **Rewards and promotions.** C24 Bank sits inside the Check24 ecosystem:

    points, vouchers, bonus conditions. Fits `fees_pricing` awkwardly at best.

 4. **Ads and upselling.** Pressure to upgrade to a paid plan. Currently would

    land in `feature_gap`, which is not what that label is for.



 Be strict about adding labels. Every class needs enough examples in a

 ~300-review gold set to be evaluated — under about 20 it cannot be judged, and

 your CLAUDE.md already says not to merge small classes afterwards to make the

 numbers look better. A gap seen three times is a note in the README, not a

 ninth label.

 ## 4. Load the filled sheet

In [ ]:
filled = pd.read_csv(TALLY_PATH, encoding="utf-8-sig")

# The sheet was filled against the original eight labels; labels added after it have no column.
TALLY_LABELS = [name for name in LABEL_NAMES if name in filled.columns]
print(f"sheet has {len(TALLY_LABELS)} of {len(LABEL_NAMES)} current labels")

# Fail loudly on blanks or typos rather than silently counting them as 0 or 1.
label_values = filled[TALLY_LABELS]
bad = ~label_values.isin([0, 1])
if bad.any().any():
    raise ValueError(
        f"label cells must be 0 or 1; bad cells per column:\n{bad.sum()[bad.sum() > 0]}"
    )
filled[TALLY_LABELS] = label_values.astype(bool)

labelled = filled[filled[TALLY_LABELS].any(axis=1)]
print(f"{len(labelled)} of {len(filled)} rows have at least one label")

 ## 5. Frequency and co-occurrence



 Two things decide whether the taxonomy survives:



 - **Per-class counts.** Scaled to a 300-review gold set, any class under ~20

   cannot be evaluated. Either it merges, or the gold set oversamples it.

 - **Co-occurrence.** `support_experience` is meant to co-occur — that is the

   core of the analysis. Any *other* pair that co-occurs most of the time is

   two names for one thing.

In [ ]:
counts = filled[TALLY_LABELS].sum().sort_values(ascending=False)
summary = pd.DataFrame(
    {
        "in_150": counts,
        "share": (counts / len(filled)).round(3),
        "projected_in_300": (counts / len(filled) * 300).round(0),
    }
)
print(summary)
print()
print("labels per review:")
print(filled[TALLY_LABELS].sum(axis=1).value_counts().sort_index())

In [ ]:
matrix = filled[TALLY_LABELS].astype(int)
co = matrix.T @ matrix
# Share of rows with the row-label that also carry the column-label.
conditional = co.div(pd.Series(counts).reindex(co.index), axis=0).round(2)
print("P(column | row):")
conditional

 ## 6. What did not fit

In [ ]:
if not filled["doesnt_fit"].isin([0, 1]).all():
    raise ValueError("doesnt_fit cells must be 0 or 1")

gaps = filled[filled["doesnt_fit"] == 1]
print(f"{len(gaps)} reviews flagged as not fitting ({len(gaps) / len(filled):.1%})\n")
# App matters: a gap seen for one app only argues against a new label (hypothesis 2).
print(gaps["app_key"].value_counts().to_string(), "\n")
for _, row in gaps.iterrows():
    print(f"[{row['app_key']} {row['score']}*] {row['notes'] if pd.notna(row['notes']) else '(no note)'}")

In [16]:
notes = filled[filled["notes"].notna() & (filled["notes"].astype(str).str.strip() != "")]
print(f"{len(notes)} reviews with a note\n")
for _, row in notes.iterrows():
    print(f"[{row['score']}*] {row['notes']}")


85 reviews with a note

[2*] repeat security check on an EXISTING account; borderline kyc vs. account-restriction
[1*] contentless negative: rejection with no topic; not praise/neutral so not no_complaint
[4*] also sluggishness (perf) - no home in taxonomy
[3*] existing payment-recall/block function broken
[3*] cancellation only by phone; process gap, filed as missing in-app function
[3*] app size/storage footprint
[3*] H2 crypto alerts + H3 cashback/points comparison
[2*] also 'quaelend langsam' (perf)
[1*] no overdraft/credit = missing feature
[1*] H3: signup premium never paid out - core complaint has no label
[1*] forced update; not a bug, so not crash_stability
[2*] H4: 'Werbung ohne Ende'; also unauthorised debits
[3*] H4: too much advertising inside the app
[3*] cash-deposit flow buggy; also load times (perf)
[3*] transactions not updating in real time
[1*] cancellation blocked in app
[1*] video re-ident to lift an account restriction
[1*] H1: account closed by the bank mid-rela

 ## 7. Then



 1. Edit `src/support_reply_gap/labeling/taxonomy.py`: adjust wording, add or

    merge labels if the evidence above justifies it, and paste two or three

    real German examples per label.

 2. `python -m support_reply_gap.labeling` to regenerate `docs/taxonomy.md`.

 3. Commit both. The taxonomy is frozen from that commit until the gold set is

    finished — a change afterwards means relabelling.